# Beach Human Detection — CPU Resume Notebook (Epoch 89 → 100)

This notebook resumes the existing **BHD batch-10 experiment** from its Google Drive recovery checkpoint. It is configured to use **CPU only** and to continue from the saved epoch, expected to be epoch 89.

- Model: Faster R-CNN ResNet-50 FPN v2
- Foreground classes: `1 = Human on Land`, `2 = Human in Water` (Torchvision labels; `0` is background)
- Batch size: 10, matching the checkpoint configuration
- Total target epochs: 100
- Checkpoints remain in `MyDrive/BHD_batch10_preprocessing_experiment/checkpoints`

**Important:** CPU training can be very slow and batch size 10 may need substantial RAM. This notebook preserves the saved batch size for consistency. If CPU RAM runs out, stop and tell me before changing the batch size. Do not delete or overwrite the checkpoint manually.

## 1. Install/import dependencies

In [ ]:
!pip -q install roboflow opencv-python-headless pyyaml

import os, random, json, time, math, warnings
from pathlib import Path
from getpass import getpass

import numpy as np
import cv2
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.8 MB/s eta 0:00:00
PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu


## 2. Mount Google Drive and locate the saved checkpoint

In [ ]:
from google.colab import drive

MOUNT_PATH = "/content/gdrive"
if Path(MOUNT_PATH, "MyDrive").exists():
    print("Google Drive already appears mounted at", MOUNT_PATH)
else:
    Path(MOUNT_PATH).mkdir(parents=True, exist_ok=True)
    drive.mount(MOUNT_PATH)

OUT = Path("/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment")
CHECKPOINT_DIR = OUT / "checkpoints"
LATEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_last.pth"
BEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_best.pth"
HISTORY_PATH = OUT / "training_history.json"

print("Experiment folder exists:", OUT.exists())
print("Latest checkpoint exists:", LATEST_CKPT.exists())
print("Best checkpoint exists:", BEST_CKPT.exists())

if not LATEST_CKPT.exists():
    raise FileNotFoundError(
        f"Expected checkpoint not found: {LATEST_CKPT}\n"
        "Check that the correct Google account is mounted and the checkpoint is in Drive."
    )

Mounted at /content/gdrive
Experiment folder exists: True
Latest checkpoint exists: True
Best checkpoint exists: True


## 3. Configure CPU-only training and inspect checkpoint

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")  # Explicitly force CPU
NUM_EPOCHS = 100
BATCH_SIZE = 10
NUM_WORKERS = 0               # Safer for Colab CPU sessions
LEARNING_RATE = 0.005
CONFIDENCE_THRESHOLD = 0.30

# Load checkpoint on CPU only to inspect its epoch and configuration.
checkpoint_preview = torch.load(LATEST_CKPT, map_location="cpu", weights_only=False)
saved_epoch_index = int(checkpoint_preview["epoch"])
saved_completed_epochs = saved_epoch_index + 1
start_epoch = saved_epoch_index + 1

print("Device:", DEVICE)
print("Checkpoint:", LATEST_CKPT)
print("Last completed epoch:", saved_completed_epochs)
print("Next epoch to run:", start_epoch + 1)
print("Saved history entries:", len(checkpoint_preview.get("history", {}).get("train_loss", [])))
print("Saved batch size:", checkpoint_preview.get("batch_size", "not recorded"))

if saved_completed_epochs >= NUM_EPOCHS:
    print("Checkpoint already completed the requested 100 epochs; no training loop is needed.")
elif saved_completed_epochs != 88:
    print("NOTE: checkpoint is not at epoch 88; the notebook will resume from the epoch recorded in the checkpoint.")

Device: cpu
Checkpoint: /content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth
Last completed epoch: 88
Next epoch to run: 89
Saved history entries: 88
Saved batch size: 10


## 4. Download the same Roboflow dataset

Use the same Roboflow project and version as the original experiment. Enter your **rotated/current** API key when prompted; the key is hidden while typing.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("image-analysis-project-rfnyq").project("bhd-m9qqy")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = Path(dataset.location)
print("Dataset location:", DATASET_DIR)
print("data.yaml exists:", (DATASET_DIR / "data.yaml").exists())

Roboflow API key: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bhd-1 in yolov8:: 100%|██████████| 479/479 [00:00<00:00, 7171.13it/s]

Dataset location: /content/bhd-1
data.yaml exists: True


## 5. Verify class mapping and locate data splits

In [ ]:
with open(DATASET_DIR / "data.yaml", "r") as f:
    CFG = yaml.safe_load(f)

raw_names = CFG.get("names", {})
if isinstance(raw_names, list):
    names = {i: str(v) for i, v in enumerate(raw_names)}
else:
    names = {int(k): str(v) for k, v in raw_names.items()}

print("Classes declared in data.yaml:")
for k, v in sorted(names.items()):
    print(f"  Source ID {k}: {v}")

# Keep the same mapping used by the saved experiment.
SOURCE_LAND_ID = 0
SOURCE_WATER_ID = 1
if SOURCE_LAND_ID == SOURCE_WATER_ID:
    raise ValueError("Land and water source IDs must differ.")
if SOURCE_LAND_ID not in names or SOURCE_WATER_ID not in names:
    raise ValueError(f"Expected source IDs 0 and 1 not found: {names}")

SOURCE_TO_SEMANTIC = {SOURCE_LAND_ID: 0, SOURCE_WATER_ID: 1}
SEMANTIC_NAMES = {0: "Human on Land", 1: "Human in Water"}
print("Source mapping:", SOURCE_TO_SEMANTIC)
print("Please verify this class order matches the original training run.")

def find_split(name):
    candidates = [DATASET_DIR / name / "images", DATASET_DIR / name]
    for p in candidates:
        if p.exists() and p.is_dir():
            if any(x.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"} for x in p.iterdir() if x.is_file()):
                return p
    return None

TRAIN_DIR = find_split("train")
VAL_DIR = find_split("valid") or find_split("val")
if TRAIN_DIR is None:
    raise FileNotFoundError(f"Could not find train images under {DATASET_DIR}")
if VAL_DIR is None:
    print("WARNING: no validation split found; validation loss will be skipped.")
print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)

Classes declared in data.yaml:
  Source ID 0: 0
  Source ID 1: 1
Source mapping: {0: 0, 1: 1}
Please verify this class order matches the original training run.
Train: /content/bhd-1/train/images
Validation: /content/bhd-1/valid/images


## 6. Build the dataset and loaders

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
METHODS = ["clahe", "gaussian", "median", "bilateral"]

def paired_label_path(image_path, split_dir):
    if image_path.parent.name == "images":
        label_dir = image_path.parent.parent / "labels"
    else:
        label_dir = split_dir / "labels"
    return label_dir / f"{image_path.stem}.txt"

def preprocess_rgb(rgb, method):
    if method == "original":
        return rgb
    if method == "clahe":
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    if method == "gaussian":
        return cv2.GaussianBlur(rgb, (5, 5), 0)
    if method == "median":
        return cv2.medianBlur(rgb, 5)
    if method == "bilateral":
        return cv2.bilateralFilter(rgb, 9, 75, 75)
    raise ValueError(f"Unknown preprocessing method: {method}")

class BHDDataset(Dataset):
    def __init__(self, split_dir, training=False, method="original"):
        self.split_dir = Path(split_dir)
        self.training = training
        self.method = method
        image_root = self.split_dir / "images" if (self.split_dir / "images").exists() else self.split_dir
        self.images = sorted(p for p in image_root.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        if not self.images:
            raise FileNotFoundError(f"No images found in {image_root}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = self.images[idx]
        rgb = np.array(Image.open(path).convert("RGB"))
        method = random.choice(METHODS) if self.training else self.method
        rgb = preprocess_rgb(rgb, method)
        h, w = rgb.shape[:2]
        label_path = paired_label_path(path, self.split_dir)
        boxes, labels = [], []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                source_id = int(float(parts[0]))
                if source_id not in SOURCE_TO_SEMANTIC:
                    continue
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = max(0.0, (xc - bw / 2) * w)
                y1 = max(0.0, (yc - bh / 2) * h)
                x2 = min(float(w), (xc + bw / 2) * w)
                y2 = min(float(h), (yc + bh / 2) * h)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(SOURCE_TO_SEMANTIC[source_id] + 1)
        img = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float() / 255.0
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
            "path": str(path),
        }
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = BHDDataset(TRAIN_DIR, training=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = None
if VAL_DIR is not None:
    val_ds = BHDDataset(VAL_DIR, training=False, method="original")
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print("Training images:", len(train_ds))
print("Batches per epoch:", len(train_loader))
print("Batch size:", BATCH_SIZE)

Training images: 166
Batches per epoch: 17
Batch size: 10


## 7. Recreate the model, optimizer and scheduler, then restore checkpoint

In [ ]:

import torch
from pathlib import Path
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# --------------------------------------------------
# 1. Configure CPU and checkpoint paths
# --------------------------------------------------
DEVICE = torch.device("cpu")

OUT = Path("/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment")
CHECKPOINT_DIR = OUT / "checkpoints"
LATEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_last.pth"
BEST_CKPT = CHECKPOINT_DIR / "fasterrcnn_best.pth"
HISTORY_PATH = OUT / "training_history.json"

if not LATEST_CKPT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {LATEST_CKPT}\n"
        "Make sure Google Drive is mounted at /content/gdrive."
    )

# --------------------------------------------------
# 2. Load the saved checkpoint
# --------------------------------------------------
checkpoint = torch.load(
    LATEST_CKPT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded.")
print("Last completed epoch:", int(checkpoint["epoch"]) + 1)

# --------------------------------------------------
# 3. Recreate the exact model architecture
#    3 classes: background, land, water
# --------------------------------------------------
model = fasterrcnn_resnet50_fpn_v2(
    weights=None,
    weights_backbone=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features, 3
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)

print("Saved model weights restored on CPU.")

# --------------------------------------------------
# 4. Create a fresh optimizer and scheduler
# --------------------------------------------------
LEARNING_RATE = 0.001  # Use your notebook's original value if different

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.1
)

# --------------------------------------------------
# 5. Restore epoch and training history
# --------------------------------------------------
start_epoch = int(checkpoint["epoch"]) + 1
best_val = float(checkpoint.get("best_val", float("inf")))

history = checkpoint.get(
    "history",
    {"train_loss": [], "val_loss": []}
)

# Ensure expected history keys exist
history.setdefault("train_loss", [])
history.setdefault("val_loss", [])

NUM_EPOCHS = 100
BATCH_SIZE = 10

print("\nResume setup complete.")
print("Next epoch number:", start_epoch + 1)
print("Target final epoch:", NUM_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Device:", DEVICE)
print("Optimizer state: freshly initialized")


Checkpoint loaded.
Last completed epoch: 88
Saved model weights restored on CPU.

Resume setup complete.
Next epoch number: 89
Target final epoch: 100
Batch size: 10
Device: cpu
Optimizer state: freshly initialized


In [ ]:
import torch
import torchvision
import numpy as np
import json
import os
from pathlib import Path

print("PyTorch:", torch.__version__)
print("Device: CPU")

PyTorch: 2.11.0+cpu
Device: CPU


In [ ]:
import torch
from pathlib import Path

LATEST_CKPT = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

if not LATEST_CKPT.exists():
    raise FileNotFoundError(f"Checkpoint not found: {LATEST_CKPT}")

checkpoint = torch.load(
    LATEST_CKPT,
    map_location="cpu",
    weights_only=False
)

# Restore training progress
start_epoch = int(checkpoint["epoch"]) + 1
NUM_EPOCHS = 100

history = checkpoint.get(
    "history",
    {"train_loss": [], "val_loss": []}
)
history.setdefault("train_loss", [])
history.setdefault("val_loss", [])

best_val = float(checkpoint.get("best_val", float("inf")))

print("Checkpoint loaded.")
print("Completed epochs:", start_epoch)
print("Next epoch:", start_epoch + 1)
print("Target epochs:", NUM_EPOCHS)
print("Remaining epochs:", max(0, NUM_EPOCHS - start_epoch))

Checkpoint loaded.
Completed epochs: 88
Next epoch: 89
Target epochs: 100
Remaining epochs: 12


In [ ]:
import time
print("time module imported successfully")

time module imported successfully


In [ ]:

import torch
from pathlib import Path
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# CPU configuration
DEVICE = torch.device("cpu")

# Existing checkpoint path
LATEST_CKPT = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

if not LATEST_CKPT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {LATEST_CKPT}. Check that Google Drive is mounted."
    )

# Load saved checkpoint
checkpoint = torch.load(
    LATEST_CKPT,
    map_location="cpu",
    weights_only=False
)

# Recreate the model architecture: background + 2 foreground classes
model = fasterrcnn_resnet50_fpn_v2(
    weights=None,
    weights_backbone=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 3)

# Restore trained weights
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

print("Model restored successfully on CPU.")
print("Completed epochs:", int(checkpoint["epoch"]) + 1)
print("Next epoch:", int(checkpoint["epoch"]) + 2)


Model restored successfully on CPU.
Completed epochs: 88
Next epoch: 89


In [ ]:
for name in [
    "train_dataset",
    "val_dataset",
    "train_loader",
    "val_loader",
    "model",
    "optimizer",
    "scheduler",
]:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

train_dataset: MISSING
val_dataset: MISSING
train_loader: MISSING
val_loader: MISSING
model: READY
optimizer: MISSING
scheduler: MISSING


In [ ]:
from pathlib import Path

search_roots = [
    Path("/content"),
    Path("/content/gdrive/MyDrive")
]

print("Searching for dataset configuration files...")
for root in search_roots:
    if root.exists():
        for yaml_file in root.rglob("data.yaml"):
            print("Found:", yaml_file)

print("\nSearching for train/valid/test image folders...")
for root in search_roots:
    if root.exists():
        for folder in root.rglob("images"):
            if folder.is_dir() and folder.parent.name.lower() in [
                "train", "valid", "val", "test"
            ]:
                print("Found:", folder)

Searching for dataset configuration files...
Found: /content/bhd-1/data.yaml
Found: /content/gdrive/MyDrive/bhc/data.yaml
Found: /content/gdrive/MyDrive/bhc/data.yaml

Searching for train/valid/test image folders...
Found: /content/bhd-1/valid/images
Found: /content/bhd-1/test/images
Found: /content/bhd-1/train/images
Found: /content/gdrive/.Encrypted/MyDrive/bhc/test/images
Found: /content/gdrive/.Encrypted/MyDrive/bhc/valid/images
Found: /content/gdrive/.Encrypted/MyDrive/bhc/train/images
Found: /content/gdrive/MyDrive/bhc/test/images
Found: /content/gdrive/MyDrive/bhc/valid/images
Found: /content/gdrive/MyDrive/bhc/train/images
Found: /content/gdrive/MyDrive/bhc/test/images
Found: /content/gdrive/MyDrive/bhc/valid/images
Found: /content/gdrive/MyDrive/bhc/train/images


In [ ]:
from pathlib import Path
import yaml

DATA_DIR = Path("/content/bhd-1")

# Read dataset class mapping
yaml_path = DATA_DIR / "data.yaml"
with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("DATA CONFIG:")
print(data_config)

# Check each dataset split
for split in ["train", "valid", "test"]:
    split_dir = DATA_DIR / split
    images_dir = split_dir / "images"
    labels_dir = split_dir / "labels"

    image_count = len([
        p for p in images_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]) if images_dir.exists() else 0

    label_count = len(list(labels_dir.glob("*.txt"))) if labels_dir.exists() else 0

    print(f"\n{split}:")
    print("Images:", image_count)
    print("Label files:", label_count)
    print("Images folder exists:", images_dir.exists())
    print("Labels folder exists:", labels_dir.exists())

    if labels_dir.exists():
        sample = next(labels_dir.glob("*.txt"), None)
        if sample:
            print("Sample annotation file:", sample.name)
            print("Sample contents:")
            print(sample.read_text()[:500] or "(empty)")

DATA CONFIG:
{'names': ['0', '1'], 'nc': 2, 'roboflow': {'license': 'CC BY 4.0', 'project': 'bhd-m9qqy', 'url': 'https://universe.roboflow.com/image-analysis-project-rfnyq/bhd-m9qqy/dataset/1', 'version': 1, 'workspace': 'image-analysis-project-rfnyq'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}

train:
Images: 166
Label files: 166
Images folder exists: True
Labels folder exists: True
Sample annotation file: frame_0235_jpg.rf.87c1a82cd64a81bfa0697622a5903fa8.txt
Sample contents:
1 0.09296875 0.8265625 0.02265625 0.08203125
1 0.1375 0.78359375 0.02265625 0.05078125
1 0.234375 0.78515625 0.02109375 0.0484375
1 0.265625 0.8296875 0.03125 0.06484375
0 0.6203125 0.58828125 0.05078125 0.04921875
0 0.70078125 0.64921875 0.028125 0.16328125
0 0.78828125 0.58828125 0.03046875 0.16875
0 0.61875 0.48984375 0.01484375 0.071875
0 0.640625 0.48828125 0.01875 0.08828125
0 0.6125 0.42734375 0.01171875 0.053125
0 0.659375 0.45 0.01484375 0.0390625
0 0.58125 0.399218

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import torchvision.transforms.functional as TF
import random

DATA_DIR = Path("/content/bhd-1")
BATCH_SIZE = 10

# Preserve class IDs:
# YOLO 0 -> Faster R-CNN label 1
# YOLO 1 -> Faster R-CNN label 2
# Faster R-CNN label 0 is background.

class BHCDataset(Dataset):
    def __init__(self, split):
        self.image_dir = DATA_DIR / split / "images"
        self.label_dir = DATA_DIR / split / "labels"

        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.images = sorted(
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in extensions
        )

        if not self.images:
            raise RuntimeError(f"No images found in {self.image_dir}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        label_path = self.label_dir / f"{image_path.stem}.txt"
        boxes, labels = [], []

        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                class_id, xc, yc, bw, bh = map(float, parts)
                class_id = int(class_id)

                if class_id not in (0, 1):
                    continue

                # YOLO normalized center coordinates -> pixel corners
                x1 = (xc - bw / 2) * width
                y1 = (yc - bh / 2) * height
                x2 = (xc + bw / 2) * width
                y2 = (yc + bh / 2) * height

                # Clamp boxes to image boundaries
                x1 = max(0, min(x1, width))
                y1 = max(0, min(y1, height))
                x2 = max(0, min(x2, width))
                y2 = max(0, min(y2, height))

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(class_id + 1)

        image = TF.to_tensor(image)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "area": torch.tensor(
                [(b[2] - b[0]) * (b[3] - b[1]) for b in boxes],
                dtype=torch.float32
            ),
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
        }

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = BHCDataset("train")
val_dataset = BHCDataset("valid")
test_dataset = BHCDataset("test")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print("Dataset and DataLoaders created successfully.")
print("Train:", len(train_dataset), "images;", len(train_loader), "batches")
print("Validation:", len(val_dataset), "images;", len(val_loader), "batches")
print("Test:", len(test_dataset), "images;", len(test_loader), "batches")

Dataset and DataLoaders created successfully.
Train: 166 images; 17 batches
Validation: 47 images; 5 batches
Test: 24 images; 3 batches


In [ ]:
images, targets = next(iter(train_loader))

print("Image batch size:", len(images))
print("First image shape:", images[0].shape)
print("First target boxes:", targets[0]["boxes"].shape)
print("First target labels:", targets[0]["labels"].unique().tolist())

print("\nStatus:")
for name in [
    "train_dataset", "val_dataset", "train_loader",
    "val_loader", "test_loader", "model"
]:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

Image batch size: 10
First image shape: torch.Size([3, 640, 640])
First target boxes: torch.Size([21, 4])
First target labels: [1, 2]

Status:
train_dataset: READY
val_dataset: READY
train_loader: READY
val_loader: READY
test_loader: READY
model: READY


In [ ]:
import torch
import torch.optim as optim
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# Restore the saved model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

# Resume after the last saved epoch
start_epoch = int(checkpoint["epoch"]) + 1
NUM_EPOCHS = 100

# Fresh optimizer because the previous optimizer state was incompatible
optimizer = optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# Fresh scheduler; its previous state is not restored
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

print("Checkpoint loaded:", CHECKPOINT_PATH.name)
print("Checkpoint epoch value:", checkpoint["epoch"])
print("Next training epoch:", start_epoch + 1)
print("Target final epoch:", NUM_EPOCHS)
print("Device:", DEVICE)
print("Optimizer:", type(optimizer).__name__)
print("Scheduler:", type(scheduler).__name__)

Checkpoint loaded: fasterrcnn_last.pth
Checkpoint epoch value: 87
Next training epoch: 89
Target final epoch: 100
Device: cpu
Optimizer: SGD
Scheduler: StepLR


In [ ]:
for name in [
    "run_train_epoch",
    "run_validation",
    "train_loader",
    "val_loader",
    "optimizer",
    "scheduler",
    "start_epoch",
    "DEVICE",
]:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

run_train_epoch: READY
run_validation: MISSING
train_loader: READY
val_loader: READY
optimizer: READY
scheduler: READY
start_epoch: READY
DEVICE: READY


In [ ]:
import torch

def run_validation(model, loader, device=DEVICE):
    """
    Calculate average validation loss without updating model weights.
    """
    model.train()  # Faster R-CNN returns loss values in training mode
    total_loss = 0.0
    batches = 0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [
                {key: value.to(device) for key, value in target.items()}
                for target in targets
            ]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if torch.isfinite(losses):
                total_loss += losses.item()
                batches += 1
            else:
                print("Warning: Non-finite validation loss; batch skipped.")

    model.train()

    if batches == 0:
        raise RuntimeError("No valid validation batches were processed.")

    return total_loss / batches


print("run_validation: READY")

run_validation: READY


In [ ]:
for name in [
    "run_train_epoch",
    "run_validation",
    "train_loader",
    "val_loader",
    "optimizer",
    "scheduler",
    "start_epoch",
    "DEVICE",
    "model",
]:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

run_train_epoch: READY
run_validation: READY
train_loader: READY
val_loader: READY
optimizer: READY
scheduler: READY
start_epoch: READY
DEVICE: READY
model: READY


In [ ]:
from tqdm.auto import tqdm
import time

print("tqdm and time: READY")

tqdm and time: READY


In [ ]:
import torch
import time
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Imports ready.")

PyTorch version: 2.11.0+cpu
CUDA available: False
Imports ready.


In [ ]:
import torch
from pathlib import Path

DEVICE = torch.device("cpu")
NUM_EPOCHS = 100

CHECKPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Check that Google Drive is mounted at /content/gdrive."
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

start_epoch = int(checkpoint["epoch"]) + 1

print("Checkpoint loaded:", CHECKPOINT_PATH.name)
print("Completed epochs:", len(checkpoint.get("history", [])))
print("Next epoch:", start_epoch + 1)
print("Target epoch:", NUM_EPOCHS)
print("Device:", DEVICE)

Checkpoint loaded: fasterrcnn_last.pth
Completed epochs: 2
Next epoch: 89
Target epoch: 100
Device: cpu


In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pathlib import Path

DEVICE = torch.device("cpu")

CHECKPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

# Load the saved checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# Recreate Faster R-CNN with background + 2 classes
model = fasterrcnn_resnet50_fpn_v2(
    weights=None,
    weights_backbone=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 3)

# Restore trained weights
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

# Restore epoch information
NUM_EPOCHS = 100
start_epoch = int(checkpoint["epoch"]) + 1

print("Model restored successfully.")
print("Checkpoint:", CHECKPOINT_PATH.name)
print("Completed epochs:", len(checkpoint.get("history", [])))
print("Next epoch:", start_epoch + 1)
print("Target epoch:", NUM_EPOCHS)
print("Device:", DEVICE)

Model restored successfully.
Checkpoint: fasterrcnn_last.pth
Completed epochs: 2
Next epoch: 89
Target epoch: 100
Device: cpu


In [ ]:
required = [
    "model", "optimizer", "scheduler",
    "train_loader", "val_loader",
    "run_train_epoch", "calculate_val_loss",
    "start_epoch", "NUM_EPOCHS", "DEVICE", "time", "tqdm"
]

for name in required:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

model: READY
optimizer: MISSING
scheduler: MISSING
train_loader: MISSING
val_loader: MISSING
run_train_epoch: READY
calculate_val_loss: READY
start_epoch: READY
NUM_EPOCHS: READY
DEVICE: READY
time: READY
tqdm: READY


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import torchvision.transforms.functional as TF

DEVICE = torch.device("cpu")
BATCH_SIZE = 10
DATA_DIR = Path("/content/bhd-1")
CHECKPOINT_DIR = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints"
)

CHECKPOINT_PATH = CHECKPOINT_DIR / "fasterrcnn_last.pth"

assert DATA_DIR.exists(), f"Dataset missing: {DATA_DIR}"
assert CHECKPOINT_PATH.exists(), f"Checkpoint missing: {CHECKPOINT_PATH}"

class BHCDataset(Dataset):
    def __init__(self, split):
        self.image_dir = DATA_DIR / split / "images"
        self.label_dir = DATA_DIR / split / "labels"
        extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        self.images = sorted(
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in extensions
        )
        if not self.images:
            raise RuntimeError(f"No images found: {self.image_dir}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = self.images[idx]
        image = Image.open(path).convert("RGB")
        width, height = image.size
        boxes, labels = [], []

        label_path = self.label_dir / f"{path.stem}.txt"
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue

                class_id, xc, yc, bw, bh = map(float, parts)
                class_id = int(class_id)
                if class_id not in (0, 1):
                    continue

                x1 = max(0, min((xc - bw / 2) * width, width))
                y1 = max(0, min((yc - bh / 2) * height, height))
                x2 = max(0, min((xc + bw / 2) * width, width))
                y2 = max(0, min((yc + bh / 2) * height, height))

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(class_id + 1)

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)

        target = {
            "boxes": boxes,
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) *
                    (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
        }

        return TF.to_tensor(image), target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = BHCDataset("train")
val_dataset = BHCDataset("valid")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

# Fresh optimizer: momentum state is not restored from the old checkpoint.
optimizer = optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Optimizer and scheduler: READY")

Training images: 166
Validation images: 47
Training batches: 17
Validation batches: 5
Optimizer and scheduler: READY


In [ ]:
required = [
    "model", "optimizer", "scheduler",
    "train_loader", "val_loader",
    "run_train_epoch", "calculate_val_loss",
    "start_epoch", "NUM_EPOCHS", "DEVICE", "time", "tqdm"
]

for name in required:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

model: READY
optimizer: READY
scheduler: READY
train_loader: READY
val_loader: READY
run_train_epoch: READY
calculate_val_loss: READY
start_epoch: READY
NUM_EPOCHS: READY
DEVICE: READY
time: READY
tqdm: READY


In [ ]:
import torch
import torchvision
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
from pathlib import Path
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import time
import os

print("All imports loaded successfully.")
print("PyTorch:", torch.__version__)

All imports loaded successfully.
PyTorch: 2.11.0+cpu


In [ ]:
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}. Check your Google Drive mount."
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

NUM_EPOCHS = 100
start_epoch = int(checkpoint["epoch"]) + 1

print("Checkpoint loaded successfully")
print("Completed epochs recorded:", len(checkpoint.get("history", [])))
print("Next epoch to train:", start_epoch + 1)
print("Final target epoch:", NUM_EPOCHS)

Checkpoint loaded successfully
Completed epochs recorded: 2
Next epoch to train: 89
Final target epoch: 100


In [ ]:
import torch
from pathlib import Path
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE = torch.device("cpu")

CHECKPOINT_PATH = Path(
    "/content/gdrive/MyDrive/BHD_batch10_preprocessing_experiment/checkpoints/fasterrcnn_last.pth"
)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

model = fasterrcnn_resnet50_fpn_v2(
    weights=None,
    weights_backbone=None
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 3)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

NUM_EPOCHS = 100
start_epoch = int(checkpoint["epoch"]) + 1

print("Model restored:", "READY")
print("Completed epochs:", len(checkpoint.get("history", [])))
print("Next epoch:", start_epoch + 1)

Model restored: READY
Completed epochs: 2
Next epoch: 89


In [ ]:
for name in [
    "model", "optimizer", "scheduler",
    "train_loader", "val_loader",
    "run_train_epoch", "calculate_val_loss",
    "start_epoch", "NUM_EPOCHS"
]:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

model: READY
optimizer: READY
scheduler: READY
train_loader: READY
val_loader: READY
run_train_epoch: READY
calculate_val_loss: READY
start_epoch: READY
NUM_EPOCHS: READY


## 8. Resume training and save after every epoch

This loop starts from the epoch stored in the checkpoint. For the checkpoint verified in your session, it should start at **89/100**. Each finished epoch overwrites the recovery checkpoint in the same experiment folder, allowing further recovery.

In [ ]:
def run_train_epoch(model, loader, optimizer, epoch):
    model.train()
    running = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, targets in pbar:
        images = [im.to(DEVICE) for im in images]
        targets_device = [{
            k: (v.to(DEVICE) if torch.is_tensor(v) else v)
            for k, v in t.items() if k != "path"
        } for t in targets]
        loss_dict = model(images, targets_device)
        loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch+1}: {loss_dict}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        running += float(loss.detach().cpu())
        pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")
    return running / max(1, len(loader))

@torch.no_grad()
def calculate_val_loss(model, loader):
    if loader is None:
        return float("nan")
    was_training = model.training
    model.train()  # Torchvision detection models return losses in train mode
    total, count = 0.0, 0
    for images, targets in tqdm(loader, desc="Validation", leave=False):
        images = [im.to(DEVICE) for im in images]
        targets_device = [{
            k: (v.to(DEVICE) if torch.is_tensor(v) else v)
            for k, v in t.items() if k != "path"
        } for t in targets]
        loss_dict = model(images, targets_device)
        loss = sum(loss_dict.values())
        if torch.isfinite(loss):
            total += float(loss.cpu())
            count += 1
    model.train(was_training)
    return total / max(1, count) if count else float("nan")

if start_epoch >= NUM_EPOCHS:
    print("Checkpoint already reached 100 epochs; skipping training.")
else:
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start = time.time()
        train_loss = run_train_epoch(model, train_loader, optimizer, epoch)
        val_loss = calculate_val_loss(model, val_loader)
        scheduler.step()

        history.setdefault("train_loss", []).append(train_loss)
        history.setdefault("val_loss", []).append(val_loss)

        if math.isfinite(val_loss) and val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), BEST_CKPT)

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val": best_val,
            "history": history,
            "class_names": SEMANTIC_NAMES,
            "source_to_semantic": SOURCE_TO_SEMANTIC,
            "batch_size": BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
        }
        torch.save(checkpoint, LATEST_CKPT)
        HISTORY_PATH.write_text(json.dumps(history, indent=2))

        print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, time={(time.time()-epoch_start)/60:.1f} min")
        print("Saved recovery checkpoint:", LATEST_CKPT)

    print("Training loop finished.")

Epoch 89/100:   0%|          | 0/17 [00:00<?, ?it/s]

## 9. Verify final checkpoint

In [ ]:
final_ckpt = torch.load(LATEST_CKPT, map_location="cpu", weights_only=False)
print("Last completed epoch:", int(final_ckpt["epoch"]) + 1)
print("Saved train-history entries:", len(final_ckpt.get("history", {}).get("train_loss", [])))
print("Latest checkpoint:", LATEST_CKPT)
print("Best weights:", BEST_CKPT if BEST_CKPT.exists() else "Not available")

## CPU runtime notes

- CPU training for Faster R-CNN can be much slower than GPU training; 12 epochs may take a long time.
- This notebook uses batch size 10 to match the saved experiment. If it runs out of RAM, do not delete the checkpoint. A lower batch size can be used, but it changes the training setup.
- If Colab disconnects, rerun cells 1–7 and then the training cell. It will read the latest saved checkpoint and resume from the next epoch.
- This notebook has not been run against your live Roboflow dataset here; verify the class order printed in Section 5 before training.

In [ ]:
if HISTORY_PATH.exists():
    hist = json.loads(HISTORY_PATH.read_text())
    plt.figure(figsize=(9, 5))
    plt.plot(range(1, len(hist["train_loss"])+1), hist["train_loss"], label="Train loss")
    valid_points = [(i+1, v) for i, v in enumerate(hist["val_loss"]) if math.isfinite(v)]
    if valid_points:
        plt.plot([x for x, _ in valid_points], [y for _, y in valid_points], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Faster R-CNN training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No history found yet.")


NameError: name 'HISTORY_PATH' is not defined